# Central de Deudores — first look

BCRA monthly credit register. Run the builders in `Code/build/` first.

DuckDB does the work; pandas only prints. Never `read_parquet` a whole file here.

In [ ]:
import pathlib, duckdb, pandas as pd

ROOT = pathlib.Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CENDEU = ROOT / "Data" / "processed" / "cendeu"
CROSS  = ROOT / "Data" / "crosswalks"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

con = duckdb.connect()
con.execute("SET memory_limit='4GB'")

for f in sorted(CENDEU.glob("*.parquet")):
    n = con.sql(f"select count(*) from '{f}'").fetchone()[0]
    print(f"{f.name:26s} {n:>14,} rows   {f.stat().st_size/1e6:>7,.0f} MB")
for d in sorted(p for p in CENDEU.iterdir() if p.is_dir()):
    n = con.sql(f"select count(*) from '{d}/*.parquet'").fetchone()[0]
    print(f"{d.name + '/':26s} {n:>14,} rows")

## Columns

In [ ]:
con.sql(f"describe select * from '{CENDEU}/deudores.parquet'").df()

In [ ]:
con.sql(f"select * from '{CENDEU}/deudores.parquet' limit 10").df()

## How much is in each status

1 normal · 2 low risk · 3 medium · 4 high · 5 uncollectible · 11 fully secured

In [ ]:
con.sql(f'''
    select status,
           count(*)                     as n_obs,
           count(distinct cuit)         as borrowers,
           sum(loans_ars)/1e12          as loans_ars_tn,
           median(loans_ars)            as median_ars
    from '{CENDEU}/deudores.parquet'
    group by status order by status
''').df()

## The monthly series

`lender_month` aggregates the 24-month panel by lender. Note the last month is incomplete: 109 lenders report with a one-month lag.

In [ ]:
con.sql(f'''
    select year, month,
           sum(borrowers)                                   as relationships,
           sum(npl_borrowers)                               as npl,
           round(100.0*sum(npl_borrowers)/sum(borrowers),1) as pct_npl,
           round(sum(credit_ars)/1e12,1)                    as credit_tn,
           round(100.0*sum(npl_ars)/sum(credit_ars),1)      as pct_balance_npl
    from '{CENDEU}/lender_month/*.parquet'
    where borrower_type = 'person'
    group by year, month order by year, month
''').df()

## By lender

`cendeu_lenders.csv` is the dimension table: `regulator_class` comes from the BCRA code range, `group` is ours.

In [ ]:
lenders = pd.read_csv(CROSS / "cendeu_lenders.csv")

q = con.sql(f'''
    select ent,
           sum(borrowers)     as borrowers,
           sum(npl_borrowers) as npl,
           sum(credit_ars)    as credit_ars,
           sum(npl_ars)       as npl_ars
    from '{CENDEU}/lender_month/*.parquet'
    where ym = (select max(ym) from '{CENDEU}/lender_month/*.parquet') - 1
    group by ent
''').df()

t = q.merge(lenders, on="ent", how="left")
t["npl_share_people"] = t.npl / t.borrowers
t["npl_share_ars"]    = t.npl_ars / t.credit_ars
t.sort_values("borrowers", ascending=False).head(25)

## Individuals vs firms

CUIT prefix: 20/23/24/27 are people, 30/33/34 are companies.

In [ ]:
con.sql(f'''
    select case when substr(cuit,1,2) in ('20','23','24','27') then 'person'
                when substr(cuit,1,2) in ('30','33','34')      then 'firm'
                else 'other' end                              as kind,
           count(distinct cuit) as n,
           sum(loans_ars)/1e12  as loans_ars_tn
    from '{CENDEU}/deudores.parquet'
    group by kind order by n desc
''').df()

## Lenders per borrower

This is what the register buys us: the same borrower seen by several lenders in the same month.

In [ ]:
con.sql(f'''
    select n_lenders, count(*) as borrowers
    from (select cuit, count(distinct ent) as n_lenders
          from '{CENDEU}/deudores.parquet' group by cuit)
    group by n_lenders order by n_lenders
''').df().head(15)

## Borrower attributes

Birth 1901-01-01 and death 19010101 are sentinels, not real dates.

In [ ]:
con.sql(f'''
    select sex,
           count(*) as n,
           count(case when birth_date <> '1901-01-01' then 1 end)        as with_birth,
           count(case when death_date not in ('', '19010101') then 1 end) as dead
    from '{CENDEU}/padron.parquet' group by sex
''').df()